# Simulasi Trading Bersinyal: Dari ATR Stop/Target ke Exit Berbasis Skor

**Permintaan user**: ubah notebook 01 (ranking forward-return) jadi **simulasi trading beneran**
(entry/exit/stop-loss, bukan cuma skor ranking) dengan **win rate minimal 45%**, modal **Rp1.500.000**,
hasilnya jadi daftar rekomendasi 5-10 coin yang diurutkan dari win rate tervalidasi tertinggi.

**Notebook ini jujur soal prosesnya** — beberapa percobaan awal gagal atau ternyata menyesatkan
(bias seleksi, lookahead bug) sebelum ketemu metodologi yang benar-benar tervalidasi. Ditulis
lengkap karena pelajarannya penting buat riset lanjutan, bukan cuma hasil akhirnya.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT)); sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from data_source.fear_greed import load_fear_greed_index

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

RAW = PROJECT_ROOT / "notebooks" / "data" / "raw"
STABLE = {"BFUSDUSDT", "RLUSDUSDT", "USD1USDT", "USDCUSDT", "USDEUSDT", "USDSUSDT", "UUSDT"}
GOLD = {"PAXGUSDT", "XAUTUSDT"}
DUP = {"WBETHUSDT"}
ALL_SYMS = sorted(p.stem.replace("_1d_full", "") for p in RAW.glob("*_1d_full.csv"))
UNIV = [s for s in ALL_SYMS if s not in STABLE | GOLD | DUP]
LARGE = {"BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "DOGEUSDT", "SOLUSDT", "TRXUSDT", "LINKUSDT"}
CATEGORY = {s: ("Large" if s in LARGE else "Mid") for s in UNIV}
BTC = "BTCUSDT"
COST = 0.0015
COINS_NB = [c for c in UNIV if c != BTC]

def load(sym):
    df = pd.read_csv(RAW / f"{sym}_1d_full.csv", parse_dates=[0], index_col=0)
    df.columns = [c.lower() for c in df.columns]
    return df

raw = {s: load(s) for s in UNIV}
dates = pd.date_range("2021-01-01", "2026-06-30", freq="D")
close_df = pd.DataFrame({s: raw[s]["close"].reindex(dates) for s in UNIV})
high_df = pd.DataFrame({s: raw[s]["high"].reindex(dates) for s in UNIV})
low_df = pd.DataFrame({s: raw[s]["low"].reindex(dates) for s in UNIV})
open_df = pd.DataFrame({s: raw[s]["open"].reindex(dates) for s in UNIV})
DEV, TEST = ("2021-01-01", "2023-12-31"), ("2024-01-01", "2026-06-30")

fgi = load_fear_greed_index()["fng_value"].reindex(dates).ffill()
market_ok_s = ((close_df[BTC] > close_df[BTC].rolling(100, min_periods=100).mean()) | (fgi.rolling(14).mean() > 50))

prev = close_df.shift(1)
tr_df = pd.DataFrame(np.maximum(high_df - low_df, np.maximum((high_df - prev).abs(), (low_df - prev).abs())),
                     index=dates, columns=UNIV)
atr_df = tr_df.rolling(14, min_periods=14).mean()

def rsi(s, n=14):
    d = s.diff()
    up = d.clip(lower=0).ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    dn = (-d.clip(upper=0)).ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    return 100 - 100 / (1 + up / dn)

rsi14_df, rsi7_df = close_df.apply(rsi, n=14), close_df.apply(rsi, n=7)
mom5_df, mom7_df = close_df / close_df.shift(5) - 1, close_df / close_df.shift(7) - 1
disthigh10_df = close_df / close_df.rolling(10, min_periods=10).max() - 1
maratio10_df = close_df / close_df.rolling(10, min_periods=10).mean() - 1

STRATEGIES = {
    "RSI14+RSI7": (rsi14_df.rank(axis=1, pct=True) + rsi7_df.rank(axis=1, pct=True)) / 2,
    "Momentum_5+7": (mom5_df.rank(axis=1, pct=True) + mom7_df.rank(axis=1, pct=True)) / 2,
    "DistHigh10+MARatio10": (disthigh10_df.rank(axis=1, pct=True) + maratio10_df.rank(axis=1, pct=True)) / 2,
    "RSI14 saja": rsi14_df.rank(axis=1, pct=True),
    "Semua_gabung": (rsi14_df.rank(axis=1, pct=True) + rsi7_df.rank(axis=1, pct=True) + mom7_df.rank(axis=1, pct=True)
                     + disthigh10_df.rank(axis=1, pct=True)) / 4,
}
print(f"Universe {len(UNIV)} coin, {len(STRATEGIES)} strategi kandidat.")

Universe 40 coin, 5 strategi kandidat.


## 1. Percobaan pertama: simulasi portofolio dengan ATR stop/target tetap

Entry: coin masuk top-5 skor RSI14+RSI7. Exit: stop-loss / take-profit berbasis kelipatan ATR dari
**harga entry** (gaya sama seperti V5 lama), atau batas hari maksimum. Posisi dikelola sebagai
**portofolio** (maks 5 slot bersamaan, modal dibagi rata, compounding) -- modal Rp1.500.000.

In [2]:
CLOSE, HIGH, LOW, OPEN = (df[UNIV].to_numpy(float) for df in (close_df, high_df, low_df, open_df))
ATR = atr_df[UNIV].to_numpy(float)
MARKET_OK = market_ok_s.to_numpy(bool)
DATE_IDX = {d: i for i, d in enumerate(dates)}
NB_IDX = [UNIV.index(c) for c in COINS_NB]

def simulate_portfolio(score_df, period, sl_mult, tp_mult, max_hold, max_positions=5, k_signal=5,
                       capital_usd=None, min_order=None, cost=COST, require_market_ok=False):
    idx = dates[(dates >= period[0]) & (dates <= period[1])]
    n = len(idx)
    cash = 1.0 if capital_usd is None else capital_usd
    positions, equity, trades = {}, np.empty(n), []
    for i, t in enumerate(idx):
        o = open_df.loc[t]
        risk_on = bool(market_ok_s.get(idx[i - 1], False)) if i > 0 else True
        to_close = []
        for sym, p in positions.items():
            lo, hi = low_df.at[t, sym], high_df.at[t, sym]
            held = i - p["entry_i"]; exit_px, reason = None, None
            if pd.notna(lo) and lo <= p["stop"]:
                exit_px = o[sym] if o[sym] <= p["stop"] else p["stop"]; reason = "STOP"
            elif pd.notna(hi) and hi >= p["tp"]:
                exit_px = o[sym] if o[sym] >= p["tp"] else p["tp"]; reason = "TP"
            elif held >= max_hold:
                exit_px = o[sym] if pd.notna(o[sym]) else close_df.at[t, sym]; reason = "MAKS_HARI"
            if exit_px is not None and pd.notna(exit_px):
                cash += p["units"] * exit_px * (1 - cost)
                ret = exit_px * (1 - cost) / (p["entry"] * (1 + cost)) - 1
                trades.append(dict(symbol=sym, ret=ret, reason=reason))
                to_close.append(sym)
        for sym in to_close:
            del positions[sym]
        if i > 0 and len(positions) < max_positions and (risk_on or not require_market_ok):
            s = score_df.loc[idx[i - 1], COINS_NB].dropna().sort_values(ascending=False)
            for sym in [c for c in s.head(k_signal).index if c not in positions][:max_positions - len(positions)]:
                px, a = o[sym], atr_df.at[idx[i - 1], sym]
                if pd.isna(px) or pd.isna(a) or px <= 0:
                    continue
                eq_now = cash + sum(positions[s2]["units"] * close_df.at[idx[i-1], s2] for s2 in positions
                                    if pd.notna(close_df.at[idx[i-1], s2]))
                budget = min(cash, eq_now / max_positions)
                if budget < (min_order or 0):
                    continue
                units = budget * (1 - cost) / px
                cash -= budget
                positions[sym] = dict(units=units, entry=px, stop=px - sl_mult * a, tp=px + tp_mult * a, entry_i=i)
        equity[i] = cash + sum(positions[s2]["units"] * close_df.at[t, s2] for s2 in positions if pd.notna(close_df.at[t, s2]))
    return pd.Series(equity, index=idx), pd.DataFrame(trades)

MODAL, KURS = 1_500_000, 16_300
eq, tr = simulate_portfolio(STRATEGIES["RSI14+RSI7"], TEST, 2.5, 1.0, 7, capital_usd=MODAL / KURS, min_order=5.0)
dd = (eq / eq.cummax() - 1).min()
wr = (tr.ret > 0).mean()
print(f"sl=2.5 tp=1.0 mh=7 (UJI): trades={len(tr)} win_rate={wr:.1%} modal->Rp{eq.iloc[-1]*KURS:,.0f} "
      f"({(eq.iloc[-1]*KURS/MODAL-1):+.1%}) MAX_DD={dd:+.1%}")

sl=2.5 tp=1.0 mh=7 (UJI): trades=1351 win_rate=63.4% modal->Rp13,636,179 (+809.1%) MAX_DD=-69.6%


**Kelihatan bagus** (win rate 60%+, profit ratusan persen) -- **tapi max drawdown −70% sampai
−89%**, jauh lebih parah dari sistem manapun di proyek ini (V23 portofolio cuma −15%). Dicoba dua
perbaikan:
- **Nambah diversifikasi** (5→15 posisi bersamaan): **tidak membantu**, drawdown tetap di kisaran
  sama -- 40 coin ini tetap bergerak bareng pas crash sistemik (korelasi tinggi, sama seperti temuan
  `research_signal_trading/14` & `24`).
- **Filter market H8b**: membantu sedikit (DD membaik ke −62% s/d −67%), tapi masih jauh dari aman --
  karena stop-loss yang lebar (sengaja dibuat lebar demi naikkan win rate) tetap bikin kerugian
  per-trade besar pas salah, dan filter market tidak melindungi dari crash coin-spesifik.

## 2. Pelajaran penting #1: simulasi portofolio bias seleksi

Simulasi portofolio (maks 5 slot) **tidak mengevaluasi semua sinyal** -- cuma yang kebetulan dapat
slot kosong (antre). Dari 2855 sinyal yang muncul di periode uji, cuma ~1351 yang benar-benar jadi
trade. Ini bisa bias: sinyal yang "kebetulan dapat slot" belum tentu representasi jujur dari "kalau
sinyal ini muncul, seberapa besar peluang benernya" -- yang justru itu yang dibutuhkan buat daftar
rekomendasi per-coin.

**Perbaikan**: evaluasi **setiap sinyal secara independen** (bukan portofolio bersama) -- metodologi
yang lebih tepat buat "per-signal win rate", meski lebih pesimis.

In [3]:
def eval_signal(score_df, period, sl_mult, tp_mult, max_hold, k_signal=5, require_market_ok=True):
    '''Tiap kejadian coin masuk top-K -> satu trade independen (bukan berbagi slot portofolio).'''
    d0, d1 = DATE_IDX[pd.Timestamp(period[0])], DATE_IDX[pd.Timestamp(period[1])]
    score_arr = score_df[UNIV].to_numpy(float)
    rets, reasons = [], []
    for i in range(d0, min(d1, len(dates) - 2)):
        if require_market_ok and not MARKET_OK[i]:
            continue
        row = score_arr[i, NB_IDX]
        valid = ~np.isnan(row)
        if valid.sum() < k_signal:
            continue
        order = np.argsort(np.where(valid, row, -np.inf))[::-1][:k_signal]
        for jj in order:
            ci = NB_IDX[jj]
            entry_px, a = OPEN[i + 1, ci], ATR[i, ci]
            if np.isnan(entry_px) or np.isnan(a) or entry_px <= 0:
                continue
            stop, tp = entry_px - sl_mult * a, entry_px + tp_mult * a
            exit_px, reason = None, None
            end = min(i + 1 + max_hold, CLOSE.shape[0])
            for day in range(i + 1, end):
                lo, hi, op = LOW[day, ci], HIGH[day, ci], OPEN[day, ci]
                if not np.isnan(lo) and lo <= stop:
                    exit_px = op if (day != i + 1 and not np.isnan(op) and op <= stop) else stop; reason = "STOP"; break
                if not np.isnan(hi) and hi >= tp:
                    exit_px = op if (day != i + 1 and not np.isnan(op) and op >= tp) else tp; reason = "TP"; break
            if exit_px is None:
                last = min(end - 1, CLOSE.shape[0] - 1)
                exit_px = CLOSE[last, ci]
                if np.isnan(exit_px):
                    continue
                reason = "MAKS_HARI"
            ret = exit_px * (1 - COST) / (entry_px * (1 + COST)) - 1
            rets.append(ret); reasons.append(reason)
    return pd.DataFrame(dict(ret=rets, reason=reasons))

def summarize(tr):
    if len(tr) < 20:
        return dict(n=len(tr), win_rate=np.nan, avg_win=np.nan, avg_loss=np.nan, expectancy=np.nan)
    wr = (tr.ret > 0).mean()
    aw = tr[tr.ret > 0].ret.mean() if (tr.ret > 0).any() else 0
    al = tr[tr.ret <= 0].ret.mean() if (tr.ret <= 0).any() else 0
    return dict(n=len(tr), win_rate=wr, avg_win=aw, avg_loss=al, expectancy=wr * aw + (1 - wr) * al)

print("Parameter SAMA (sl=2.5 tp=1.0 mh=7), dua metodologi:")
print("  Portofolio (bias slot) :", "win_rate~63% (lihat bagian 1)")
print("  Per-sinyal independen  :", summarize(eval_signal(STRATEGIES["RSI14+RSI7"], TEST, 2.5, 1.0, 7)))

Parameter SAMA (sl=2.5 tp=1.0 mh=7), dua metodologi:
  Portofolio (bias slot) : win_rate~63% (lihat bagian 1)
  Per-sinyal independen  : {'n': 2855, 'win_rate': np.float64(0.6003502626970227), 'avg_win': np.float64(0.0638215633207387), 'avg_loss': np.float64(-0.10139845741288131), 'expectancy': np.float64(-0.002208574562644991)}


**Beda jauh** -- metodologi yang lebih jujur ini menunjukkan ekspektasi di parameter yang sama
jadi **negatif**. Dicoba perluas pencarian (560+ kombinasi, 5 strategi indikator, SL sampai 6×ATR) --
hasil terbaik yang bisa dicapai di uji cuma mentok di sekitar **titik impas** (−0.26% s/d +0.19%),
bukan edge yang robust. Kesimpulan sementara: **ATR stop/target tetap dari harga entry bukan
mekanisme exit yang cocok** buat sinyal momentum ini -- exit-nya "terlalu cepat" memotong posisi yang
sebenarnya masih berpotensi lanjut.

## 3. Pivot: exit berbasis skor (bukan harga tetap)

User mengusulkan dua alternatif: **exit saat skor turun** (sinyal yang bawa masuk sudah pudar) atau
**trailing stop** (ambang mengejar harga, bukan tetap dari entry). Keduanya dicoba, dengan batas hari
maksimum + stop keras persen tetap sebagai jaring pengaman darurat.

⚠️ **Pelajaran penting #2 (lookahead bug)**: percobaan pertama memakai skor/ATR hari YANG SAMA
dengan hari keputusan exit -- padahal skor itu baru "diketahui" di closing hari itu, sementara exit
dieksekusi di open hari itu (sebelum closing-nya kejadian). Ini bikin hasil kelihatan luar biasa
(+28% rata-rata untung) karena strategi "mengintip" closing hari ini buat keputusan open hari ini.
Diperbaiki: skor/ATR yang dipakai harus dari **hari sebelumnya** (konsisten dengan aturan "sinyal
dari close → eksekusi open besok" yang dipakai di semua notebook proyek ini).

In [4]:
def eval_score_exit(score_df, period, score_floor, max_hold, hard_stop_pct, k_signal=5, require_market_ok=True):
    '''Exit: skor KEMARIN jatuh di bawah score_floor, ATAU maks_hari, ATAU stop keras (jaring pengaman).'''
    d0, d1 = DATE_IDX[pd.Timestamp(period[0])], DATE_IDX[pd.Timestamp(period[1])]
    score_arr = score_df[UNIV].to_numpy(float)
    rets, reasons = [], []
    for i in range(d0, min(d1, len(dates) - 2)):
        if require_market_ok and not MARKET_OK[i]:
            continue
        row = score_arr[i, NB_IDX]
        valid = ~np.isnan(row)
        if valid.sum() < k_signal:
            continue
        order = np.argsort(np.where(valid, row, -np.inf))[::-1][:k_signal]
        for jj in order:
            ci = NB_IDX[jj]
            entry_px = OPEN[i + 1, ci]
            if np.isnan(entry_px) or entry_px <= 0:
                continue
            hard_stop = entry_px * (1 - hard_stop_pct)
            exit_px, reason = None, None
            end = min(i + 1 + max_hold, CLOSE.shape[0])
            for day in range(i + 1, end):
                lo, op = LOW[day, ci], OPEN[day, ci]
                if not np.isnan(lo) and lo <= hard_stop:
                    exit_px = op if (day != i + 1 and not np.isnan(op) and op <= hard_stop) else hard_stop
                    reason = "STOP_KERAS"; break
                sc_yesterday = score_arr[day - 1, ci]   # skor KEMARIN, bukan hari ini -- hindari lookahead
                if day > i + 1 and not np.isnan(sc_yesterday) and sc_yesterday < score_floor:
                    exit_px, reason = (op if not np.isnan(op) else CLOSE[day, ci]), "SKOR_TURUN"; break
            if exit_px is None:
                last = min(end - 1, CLOSE.shape[0] - 1)
                exit_px = CLOSE[last, ci]
                if np.isnan(exit_px):
                    continue
                reason = "MAKS_HARI"
            ret = exit_px * (1 - COST) / (entry_px * (1 + COST)) - 1
            rets.append(ret); reasons.append(reason)
    return pd.DataFrame(dict(ret=rets, reason=reasons))

print("DEV: cari floor/max_hold/hard_stop terbaik (RSI14+RSI7), target win_rate>=45%\n")
rows = []
for floor in (0.2, 0.3, 0.4, 0.5):
    for mh in (10, 12, 14):
        for hs in (0.20, 0.25):
            s = summarize(eval_score_exit(STRATEGIES["RSI14+RSI7"], DEV, floor, mh, hs))
            rows.append(dict(floor=floor, max_hold=mh, hard_stop=hs, **s))
res = pd.DataFrame(rows)
qualify = res[(res.win_rate >= 0.45) & (res.expectancy > 0)].sort_values("expectancy", ascending=False)
print(f"{len(qualify)}/{len(res)} lolos win_rate>=45% & ekspektasi positif\n")
print(qualify.head(8).to_string(index=False, formatters={"win_rate": "{:.1%}".format, "avg_win": "{:+.1%}".format,
                                                          "avg_loss": "{:+.1%}".format, "expectancy": "{:+.2%}".format}))

DEV: cari floor/max_hold/hard_stop terbaik (RSI14+RSI7), target win_rate>=45%



14/24 lolos win_rate>=45% & ekspektasi positif

 floor  max_hold  hard_stop    n win_rate avg_win avg_loss expectancy
   0.2        14       0.25 2780    47.4%  +33.3%   -12.8%     +9.08%
   0.3        14       0.25 2780    46.4%  +33.1%   -12.1%     +8.88%
   0.2        14       0.20 2780    45.7%  +32.6%   -12.6%     +8.08%
   0.3        14       0.20 2780    45.1%  +32.3%   -11.9%     +8.03%
   0.2        12       0.25 2780    48.3%  +28.8%   -12.3%     +7.61%
   0.3        12       0.25 2780    46.9%  +29.0%   -11.5%     +7.49%
   0.4        12       0.25 2780    45.4%  +28.3%   -11.0%     +6.82%
   0.2        12       0.20 2780    46.8%  +28.3%   -12.1%     +6.80%


## 4. Validasi di periode uji (SEKALI) + sensitivitas parameter

In [5]:
print("UJI (2024-2026), floor=0.3 mh=14 hard_stop=0.25, RSI14+RSI7:\n")
tr_test = eval_score_exit(STRATEGIES["RSI14+RSI7"], TEST, 0.3, 14, 0.25)
s = summarize(tr_test)
print(s)
print("Rincian alasan exit:", tr_test.reason.value_counts().to_dict())

print("\nSensitivitas parameter tetangga (UJI):")
for floor in (0.2, 0.25, 0.3, 0.35, 0.4):
    for mh in (10, 12, 14):
        s2 = summarize(eval_score_exit(STRATEGIES["RSI14+RSI7"], TEST, floor, mh, 0.25))
        print(f"  floor={floor} mh={mh}: win_rate={s2['win_rate']:.1%} expectancy={s2['expectancy']:+.2%}")

UJI (2024-2026), floor=0.3 mh=14 hard_stop=0.25, RSI14+RSI7:



{'n': 2855, 'win_rate': np.float64(0.45464098073555165), 'avg_win': np.float64(0.1916436308473403), 'avg_loss': np.float64(-0.1270211361501058), 'expectancy': np.float64(0.017856926043479146)}
Rincian alasan exit: {'MAKS_HARI': 1822, 'SKOR_TURUN': 600, 'STOP_KERAS': 433}

Sensitivitas parameter tetangga (UJI):


  floor=0.2 mh=10: win_rate=45.3% expectancy=+1.27%


  floor=0.2 mh=12: win_rate=45.4% expectancy=+1.55%


  floor=0.2 mh=14: win_rate=45.0% expectancy=+1.69%
  floor=0.25 mh=10: win_rate=45.1% expectancy=+1.29%


...

  floor=0.3 mh=14: win_rate=45.5% expectancy=+1.79%
  floor=0.35 mh=10: win_rate=45.4% expectancy=+1.34%


  floor=0.35 mh=12: win_rate=45.3% expectancy=+1.66%


  floor=0.35 mh=14: win_rate=45.2% expectancy=+1.86%
  floor=0.4 mh=10: win_rate=45.4% expectancy=+1.34%


  floor=0.4 mh=12: win_rate=45.2% expectancy=+1.70%


  floor=0.4 mh=14: win_rate=45.2% expectancy=+1.92%


**Lolos, dan stabil** -- win rate & ekspektasi hampir sama di seluruh tetangga parameter
(floor 0.2-0.4, max_hold 10-14 semua memberi win rate 44.9-45.6%, ekspektasi +1.27% s/d +1.92%) --
bukan titik optimal yang rapuh/kebetulan.

## 5. Catatan jujur: per tahun (termasuk tahun bear 2022)

In [6]:
print("Per tahun, floor=0.3 mh=14 hard_stop=0.25 (gabung dev+uji, cuma buat lihat konsistensi):\n")
for yr in range(2021, 2026):
    s = summarize(eval_score_exit(STRATEGIES["RSI14+RSI7"], (f"{yr}-01-01", f"{yr}-12-31"), 0.3, 14, 0.25))
    if s["n"] < 20:
        continue
    print(f"{yr}: n={s['n']:4d} win_rate={s['win_rate']:.1%} avg_win={s['avg_win']:+.1%} avg_loss={s['avg_loss']:+.1%} expectancy={s['expectancy']:+.2%}")

Per tahun, floor=0.3 mh=14 hard_stop=0.25 (gabung dev+uji, cuma buat lihat konsistensi):

2021: n=1220 win_rate=52.2% avg_win=+53.1% avg_loss=-15.6% expectancy=+20.26%
2022: n= 130 win_rate=14.6% avg_win=+6.6% avg_loss=-14.1% expectancy=-11.11%


2023: n=1430 win_rate=44.3% avg_win=+13.8% avg_loss=-9.2% expectancy=+0.98%
2024: n=1560 win_rate=47.4% avg_win=+24.6% avg_loss=-13.4% expectancy=+4.61%
2025: n=1065 win_rate=44.0% avg_win=+11.4% avg_loss=-11.9% expectancy=-1.62%


**2022 (bear market LUNA/FTX) hasilnya buruk**: win rate cuma 14.6%, ekspektasi −11.11% --
jauh lebih jelek dari tahun lain. 2025 juga sedikit negatif (−1.62%). Filter market H8b mengurangi
**frekuensi** sinyal di tahun-tahun ini (n=130 di 2022 vs 1000+ di tahun lain) tapi tidak sepenuhnya
melindungi **kualitas** sinyal yang tetap lolos -- sinyal yang lolos filter pas bear market yang
sedang reli singkat ternyata sering gagal lagi.

**2021 dan 2024 (bull/recovery) jauh di atas rata-rata** (expectancy +20% dan +4.6%) -- menopang
angka gabungan periode uji supaya tetap terlihat baik. Headline "win rate 45.5%, ekspektasi +1.79%"
itu nyata tapi **tidak merata sepanjang waktu** -- jangan dibaca sebagai "pasti begini tiap tahun".

## Kesimpulan

**Metodologi trading bersinyal tervalidasi, dengan tiga pelajaran metodologi penting ditemukan di
sepanjang jalan** (dicatat lengkap karena berharga buat riset lanjutan):

1. **Simulasi portofolio (slot terbatas) bisa bias seleksi** -- cuma mengevaluasi sinyal yang
   kebetulan dapat modal, bukan representasi jujur tiap sinyal. Pakai evaluasi per-sinyal independen
   buat validasi "seberapa bisa dipercaya sinyal ini", bukan simulasi portofolio compounding.
2. **ATR stop/target tetap dari harga entry bukan mekanisme exit yang cocok** di sini -- begitu
   dievaluasi jujur (poin 1), ekspektasinya mentok di sekitar titik impas meski dicoba 560+ kombinasi.
3. **Lookahead bug**: skor/indikator buat keputusan exit WAJIB dari hari sebelumnya, bukan hari yang
   sama dengan eksekusi -- kesalahan ini sempat bikin hasil kelihatan luar biasa bagus (+28% rata-rata
   untung) padahal cuma "mengintip" masa depan.

### Metodologi terpilih (lolos validasi dev→uji)
- **Sinyal masuk**: top-5 skor rank-percentile **RSI(14)+RSI(7)**, filter market H8b aktif.
- **Exit**: skor coin itu jatuh di bawah **persentil 30%** (sinyal pudar) ATAU **14 hari** terlewati
  ATAU **stop keras −25%** (jaring pengaman darurat) -- mana yang lebih dulu kejadian.
- **Hasil uji** (2024-01→2026-06): **win rate 45.5%**, rata-rata untung +19.2%, rata-rata rugi
  −12.7%, **ekspektasi per sinyal +1.79%** -- stabil di seluruh tetangga parameter.

### Catatan jujur / batasan
1. **Lemah di tahun bear** (2022: win rate 14.6%, ekspektasi −11%) -- filter market mengurangi
   frekuensi sinyal tapi tidak sepenuhnya melindungi kualitasnya saat market benar-benar jatuh.
2. Angka gabungan periode uji ditopang tahun-tahun kuat (2021, 2024) -- jangan dibaca sebagai hasil
   yang rata tiap tahun.
3. Ini statistik **per-sinyal independen**, bukan simulasi portofolio compounding -- kalau user
   mengambil banyak rekomendasi sekaligus dengan modal terbatas dan tidak mendiversifikasi, hasil
   nyata bisa beda (lihat pelajaran #1).
4. Belum ada komponen likuiditas minimum / umur listing minimum / exclude pump-and-dump ekstrem.
5. **Simulasi backtest historis, bukan jaminan hasil nyata.**

### Cara dipakai jadi daftar rekomendasi
Tiap hari: hitung skor RSI14+RSI7 semua coin di universe, ambil top 5-10 yang filter market-nya
hidup, tiap rekomendasi disertai win rate tervalidasi (+45.5%, ekspektasi +1.79%/sinyal) dan level
exit-nya (skor turun di bawah 30 persentil / 14 hari / stop −25%) -- bukan janji "pasti naik",
tapi probabilitas tervalidasi yang user bisa pakai buat filter & analisis sendiri.